# Vector DB

We will use Qdrant because it's a production ready vector db that runs in docker.

Alternatives are:
- Chroma: in-process vector db (used for prototyping)
- LanceDB: embedded like Chroma, but stores data in an open columnar format (data can live on S3, readable by other data tools)
- Pgvector: postgres extension to add vectors and semantic search
- Azure AI Search: vector db on azure
- Pinecone: vector db on cloud

Gotchas:
- In a production environment calls to the embedder model and upserts into qdrant should be done in batches (remember that OpenAI's embedder API has a limit of 300k tokens)

## Install

Run Qdrant:
1. Run `docker compose -f 4_docker-compose.yml up -d`
2. Open http://localhost:6333/dashboard to verify that Qdrant is up and running
3. Run `docker compose -f 4_docker-compose.yml logs -f` to attach to logs (same logs as the ones shown by docker compose without -d)
4. Run `docker compose -f 4_docker-compose.yml down` to stop the container

Install the client:
1. Run `poetry add qdrant-client`

## Usage

### Create the collection

In [16]:
from qdrant_client import QdrantClient
from qdrant_client.models import Distance, VectorParams
COLLECTION_NAME = "recipes"

qdrant = QdrantClient(url="http://localhost:6333")

# Delete if already exists
if qdrant.collection_exists(COLLECTION_NAME):
    qdrant.delete_collection(COLLECTION_NAME)

qdrant.create_collection(
    collection_name=COLLECTION_NAME,
    vectors_config=VectorParams(
        size=1536,  # 1536 is the default size of the vector return by the embedding model text-embedding-3-small by OpenAI (Note that OpenAI states that you can downsize the vector down to 256 preserving accuracy) 
        distance=Distance.COSINE
    ), 
)

print(f"collection created: {COLLECTION_NAME}")

collection created: recipes


## Add recipes and embeddings to Qdrant 

In [42]:
from qdrant_client.models import PointStruct
from openai import OpenAI
     
recipes = [
    { "title": "Carbonara", "category":"first_dish", "text": "Carbonara. Serves 4. Ingredients: 320g spaghetti, 150g guanciale, 4 egg yolks, 100g pecorino romano, black pepper. Method: cut the guanciale into strips and fry it in a dry pan until crisp. Beat the yolks with grated pecorino and plenty of pepper. Cook the spaghetti al dente, drain and keep some pasta water. Toss off the heat with the guanciale and the egg mixture, loosening with pasta water until the sauce is smooth." },
    { "title": "Pesto alla genovese", "category":"first_dish", "text": "Pesto alla genovese. Ingredients: 50g basil, 30g pine nuts, 2 garlic cloves, 60g parmesan, 20g pecorino, 100ml extra virgin olive oil, coarse salt. Method: crush garlic and salt in a mortar, add the basil and grind with a circular motion. Add the pine nuts until creamy, then the grated cheeses. Pour in the oil slowly while mixing. Never heat pesto: it is served raw." },
    { "title": "Tiramisu", "category":"dessert", "text": "Tiramisu. Serves 6. Ingredients: 500g mascarpone, 4 eggs, 100g sugar, 300g savoiardi biscuits, cold espresso, unsweetened cocoa. Method: whip the yolks with the sugar until pale, then fold in the mascarpone. Whip the egg whites to stiff peaks and fold them in gently. Dip the biscuits in the coffee without soaking them and layer with the cream. Chill for at least 4 hours and dust with cocoa before serving." },
    { "title": "Risotto alla milanese", "category":"first_dish", "text": "Risotto alla milanese. Serves 4. Ingredients: 320g carnaroli rice, 1 sachet of saffron, 1 onion, 80g butter, 60g parmesan, beef stock, white wine. Method: soften the chopped onion in butter, toast the rice for two minutes. Deglaze with white wine. Add hot stock one ladle at a time while stirring. Halfway through, dissolve the saffron in a little stock and stir it in. Finish off the heat with cold butter and parmesan." },
    { "title": "Amatriciana", "category":"first_dish", "text": "Amatriciana. Serves 4. Ingredients: 400g bucatini, 150g guanciale, 400g peeled tomatoes, pecorino romano, chilli, white wine. Method: fry the guanciale strips until crisp, deglaze with wine and set the meat aside. Add the crushed tomatoes and chilli, cook for 15 minutes. Return the guanciale, toss with the drained bucatini and finish with grated pecorino." },
    { "title": "Panna cotta", "category":"dessert", "text": "Panna cotta. Serves 6. Ingredients: 500ml double cream, 100g sugar, 8g gelatine leaves, 1 vanilla pod. Method: soak the gelatine in cold water for 10 minutes. Heat the cream with the sugar and vanilla seeds without boiling. Squeeze the gelatine and dissolve it in the hot cream. Pour into moulds and set in the fridge for at least 5 hours." },
    { "title": "Pasta e fagioli", "category":"first_dish", "text": "Pasta e fagioli. Serves 4. Ingredients: 250g borlotti beans, 200g mixed pasta, 1 celery stick, 1 carrot, 1 onion, 2 tomatoes, rosemary, olive oil. Method: soak the beans overnight. Fry the chopped celery, carrot and onion, add the chopped tomatoes and the drained beans. Cover with water and simmer for an hour. Blend half the beans to thicken, then add the pasta and cook it directly in the soup." },
    { "title": "Cacio e pepe", "category":"first_dish", "text": "Cacio e pepe. Serves 4. Ingredients: 320g tonnarelli, 200g pecorino romano, black peppercorns. Method: toast the coarsely crushed pepper in a pan. Cook the pasta in little water so it becomes very starchy. Melt the grated pecorino with warm pasta water into a smooth cream, keeping it off the heat or it will split. Toss the pasta with the pepper and finish with the pecorino cream." }
]

# function to embed recipes
clientAI = OpenAI()
def embed(recipe):
    resp = clientAI.embeddings.create(input=f"Title: {recipe['title']} --- Text: {recipe['text']}", model="text-embedding-3-small")
    return resp.data[0].embedding

# create points to add to qdrant
points = [
    PointStruct(
        id=i,
        vector=embed(r), 
        payload=r,
    )
    for i, r in enumerate(recipes)
]

# upsert points to qdrant
qdrant.upsert(collection_name=COLLECTION_NAME, points=points)

print(f"Upserted {len(points)} recipes")

Upserted 8 recipes


## Embed the query and run the semantic search

In [40]:
def embed_query(query):
    resp = clientAI.embeddings.create(input=query, model="text-embedding-3-small")
    return resp.data[0].embedding

def semantic_search(query):
    embedded_query = embed_query(query)
    return qdrant.query_points(
                collection_name=COLLECTION_NAME,
                query=embedded_query,
                limit=3
           ).points

results = semantic_search("I would like to make a cheese pasta")
for r in results:
    print(f"{r.payload['title']} | {r.score:.4f}")

Cacio e pepe | 0.4660
Carbonara | 0.4550
Pesto alla genovese | 0.4120


## Use the results for a RAG

In [33]:
import json

user_question = "I would like to make a cheese pasta"

# retrieve related recipes using semantic search
result = semantic_search(user_question)
retrieved_recipes = [{'title': r.payload['title'], 'text': r.payload['text'], 'score': r.score} for r in result]

# creating input for llm
input = f"recipes: {json.dumps(retrieved_recipes, indent=2)} \n\n --- \n\n user question: {user_question}"
print(f"input:\n\"{input}\"")

# generate augmented llm answer
response = clientAI.responses.create(
    model="gpt-5-nano",
    instructions="you're a chef assistant, your goal is to answer the user's question. " \
                 "You will be given a list of recipes retrieved using semantic search, the user doesn't know about the semantic search. " \
                 "use only the info included in the recipe to generate the answer. " \
                 "Pick only the most related recipe and if there are no recipes or"
                 "if the recipes don't contain enough info to answer, say so explicitly instead of trying to answer with ungrounded info.",
    input=input
)

print(f"\n\noutput: \n{response.output_text}")
print(f"\nusage: {response.usage.total_tokens}")

input:
"recipes: [
  {
    "title": "Cacio e pepe",
    "text": "Cacio e pepe. Serves 4. Ingredients: 320g tonnarelli, 200g pecorino romano, black peppercorns. Method: toast the coarsely crushed pepper in a pan. Cook the pasta in little water so it becomes very starchy. Melt the grated pecorino with warm pasta water into a smooth cream, keeping it off the heat or it will split. Toss the pasta with the pepper and finish with the pecorino cream.",
    "score": 0.46595955
  },
  {
    "title": "Carbonara",
    "text": "Carbonara. Serves 4. Ingredients: 320g spaghetti, 150g guanciale, 4 egg yolks, 100g pecorino romano, black pepper. Method: cut the guanciale into strips and fry it in a dry pan until crisp. Beat the yolks with grated pecorino and plenty of pepper. Cook the spaghetti al dente, drain and keep some pasta water. Toss off the heat with the guanciale and the egg mixture, loosening with pasta water until the sauce is smooth.",
    "score": 0.45495895
  },
  {
    "title": "Pesto a

## Filtering by metadata

In a vector db, the main feature is the semantic search but metadata are very important because they let you:
- filter and categorize data to improve retrieval accuracy (hybrid search)
- enrich data with contextual info (doc name realted to the the chunk, page numbers related to the chunk, useful to make citations)
- leverage them for RBAC

Note that Metadata can be also generated by LLM-powered extractions (e.g. an LLM could decide if the recipe is a first dish, or a dessert, or another category, you could use a literal to restrict the possible values).  

In Qdrant every field (string, number, boolean, literal, geo, datetime, uuid, arrays) you put in the payload is a metadata.

Now we can ask for something with cheese filtering just the desserts

In [ ]:
from qdrant_client.models import Filter, FieldCondition, MatchValue

embedded_query = embed_query("I would like something with cheese")
category = "dessert"

results = qdrant.query_points(
    collection_name=COLLECTION_NAME,
    query=embedded_query,
    query_filter=Filter(
        must=[ # all conditions must match
            FieldCondition(key="category", match=MatchValue(value=category)),
        ],
        should=[], # at least one condition must match
        min_should=[], # at least a minimum number of conditions must match
        must_not=[] # no condition must match
    ),
    limit=3,
).points

for r in results:
    print(f"{r.payload['title']} | {r.score:.4f}")

Panna cotta | 0.1986
Tiramisu | 0.1940


Now filtering just the first_dishes

In [ ]:
category = "first_dish"

results = qdrant.query_points(
    collection_name=COLLECTION_NAME,
    query=embedded_query,
    query_filter=Filter(
        must=[
            FieldCondition(
                key="category", 
                match=MatchValue(value=category), #MatchText (full text search), MatchTextAny (full text search at least one token), MatchPhrase (full text search on phrase), MatchPrefix, MatchAny, MatchExcept
                range=None,
                geo_bounding_box=None,
                geo_radius=None,
                is_empty=None,
                is_null=None
            ), 
        ]
    ),
    limit=3,
).points

for r in results:
    print(f"{r.payload['title']} | {r.score:.4f}")

Cacio e pepe | 0.2932
Pesto alla genovese | 0.2716
Carbonara | 0.2716


## Indexing metadata

In Qdrant you can create an index built on a metadata. 

This will create a sorted data structure to store the index, like in sql indexes.

In [ ]:
from qdrant_client.models import PayloadSchemaType

qdrant.create_payload_index(
    collection_name=COLLECTION_NAME,
    field_name="category",
    field_schema=PayloadSchemaType.KEYWORD, # integer / float / geo / text (full text search) / bool / datetime / uuid
)

UpdateResult(operation_id=8, status=<UpdateStatus.COMPLETED: 'completed'>)

Qdrant recommends to use 